# [Notebook]Unihack26_Transformer_Profile
Research preparation only. Original source splits preserved. No accuracy or production checkpoint produced. Parcel3D renders stay academic-only. Group: Unihack26.

In [ ]:
from pathlib import Path
import sys,json
SOURCES={'ml/__init__.py': '', 'ml/data/__init__.py': '', 'ml/data/parcel3d_audit.py': '"""CPU-only audit of a mounted Parcel3D input. Does not copy its image archive."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport platform\nfrom collections import Counter, deque\nfrom datetime import UTC, datetime\nfrom pathlib import Path, PurePosixPath\n\nfrom PIL import Image\n\nSOURCE = "ayesha19765/parcel3d"\nORIGINAL = "https://zenodo.org/records/8032204"\nALLOWED_SPLITS = ("train", "validation", "test")\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef within_root(root: Path, relative: str) -> Path:\n    relative = relative.replace("\\\\", "/")\n    parsed = PurePosixPath(relative)\n    if parsed.is_absolute() or ".." in parsed.parts or ":" in relative:\n        raise ValueError("UNSAFE_ANNOTATION_PATH")\n    result = (root / Path(*parsed.parts)).resolve()\n    if not result.is_relative_to(root.resolve()):\n        raise ValueError("PATH_OUTSIDE_INPUT")\n    return result\n\n\ndef find_annotations(input_root: Path) -> dict[str, Path]:\n    # Kaggle can expose both legacy and owner-qualified aliases of one input.\n    # Annotation files sit above per-image directories; never walk 13,400 samples\n    # just to locate three files. Bound discovery and compare duplicate contents.\n    roots, queue, visited = [], deque([(input_root, 0)]), set()\n    while queue:\n        directory, depth = queue.popleft()\n        resolved = directory.resolve()\n        if resolved in visited:\n            continue\n        visited.add(resolved)\n        if len(visited) > 200:\n            raise RuntimeError("ANNOTATION_DISCOVERY_LIMIT")\n        if all((directory / f"parcel3d_{split}.json").is_file() for split in ALLOWED_SPLITS):\n            roots.append(directory)\n            continue\n        if depth < 5:\n            for child in sorted(directory.iterdir()):\n                if (\n                    child.name not in {*ALLOWED_SPLITS, "misc"}\n                    and not child.name.startswith(".")\n                    and child.is_dir()\n                ):\n                    queue.append((child, depth + 1))\n    if not roots:\n        raise RuntimeError("ANNOTATION_ROOT_NOT_FOUND")\n    roots.sort(key=lambda path: (len(path.parts), str(path)))\n    if len(roots) > 1:\n        digests = {\n            tuple(sha256_file(root / f"parcel3d_{split}.json") for split in ALLOWED_SPLITS)\n            for root in roots\n        }\n        if len(digests) != 1:\n            raise RuntimeError("ANNOTATION_ALIASES_HAVE_DIFFERENT_CONTENT")\n    print(f"Annotation roots: {len(roots)} equivalent mount(s); selected {roots[0]}", flush=True)\n    return {split: roots[0] / f"parcel3d_{split}.json" for split in ALLOWED_SPLITS}\n\n\ndef resolve_image(annotation_path: Path, image_record: dict, split: str) -> Path | None:\n    filename = image_record.get("file_name")\n    if not isinstance(filename, str):\n        return None\n    candidates = [\n        within_root(annotation_path.parent, filename),\n        within_root(annotation_path.parent, f"{split}/{filename}"),\n    ]\n    for candidate in candidates:\n        if candidate.is_file():\n            return candidate\n    return None\n\n\ndef audit(input_root: Path, output_root: Path, source_version: int = 1) -> dict:\n    input_root = input_root.resolve()\n    output_root = output_root.resolve()\n    if output_root.is_relative_to(input_root):\n        raise ValueError("OUTPUT_MUST_NOT_BE_INSIDE_READ_ONLY_INPUT")\n    output_root.mkdir(parents=True, exist_ok=True)\n    print("Locating Parcel3D annotation roots (bounded discovery)", flush=True)\n    annotations = find_annotations(input_root)\n    result = {\n        "schema_version": "parcel3d-audit-v2",\n        "source": SOURCE,\n        "source_version": source_version,\n        "original_record": ORIGINAL,\n        "checked_at": datetime.now(UTC).isoformat(),\n        "runtime": {"python": platform.python_version(), "platform": platform.platform()},\n        "source_kind": "SYNTHETIC",\n        "splits": {},\n        "errors": [],\n        "license": {\n            "mirror_metadata": "Apache-2.0; not authoritative for upstream assets",\n            "original_full_dataset": "Academic research only; Zenodo other-nc",\n            "textureless_obj_meshes": "CC-BY-4.0, separate grant in original record",\n            "cardboard_textures": "CC-BY-4.0, separate record 8041823",\n            "training_policy": "Full rendered-image training remains gated pending scope review.",\n            "commercial_release": "NOT_CLEARED_FOR_ORIGINAL_RENDERED_IMAGES",\n            "reference_matching_labels": "Must inspect; COCO damage labels alone do not establish cross-view correspondence or evidence sufficiency.",\n        },\n    }\n    for package in ("numpy", "Pillow", "torch", "opencv-python"):\n        try:\n            result["runtime"][package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            pass\n    group_sets = {}\n    sample_directories = set()\n    for split, path in annotations.items():\n        print(f"Auditing {split}: {path.name}", flush=True)\n        document = json.loads(path.read_text(encoding="utf-8"))\n        images = document.get("images", [])\n        labels = document.get("annotations", [])\n        record = {\n            "annotation_path": str(path.relative_to(input_root)),\n            "annotation_sha256": sha256_file(path),\n            "annotation_bytes": path.stat().st_size,\n            "top_level_keys": sorted(document),\n            "image_count": len(images),\n            "annotation_count": len(labels),\n            "categories": document.get("categories", []),\n            "image_fields": dict(Counter(k for image in images for k in image)),\n            "annotation_fields": dict(Counter(k for annotation in labels for k in annotation)),\n            "sample_records": images[:2],\n            "sample_annotations": labels[:2],\n            "samples": [],\n        }\n        ids = [image.get("id") for image in images]\n        record["duplicate_image_ids"] = len(ids) - len(set(ids))\n        group_key = next(\n            (\n                key\n                for key in ("mesh_id", "object_id", "parcel_id", "model_id")\n                if images and all(key in item for item in images)\n            ),\n            None,\n        )\n        record["explicit_group_key"] = group_key\n        record["group_split_verified"] = False\n        if group_key:\n            group_sets[split] = {str(item[group_key]) for item in images}\n        # Deterministic stratified positions; no exhaustive image read or copy.\n        positions = sorted({int(i * max(len(images) - 1, 0) / 7) for i in range(8)})\n        for index in positions if images else []:\n            image_record = images[index]\n            image_path = resolve_image(path, image_record, split)\n            sample = {\n                "image_id": image_record.get("id"),\n                "file_name": image_record.get("file_name"),\n            }\n            if image_path is None:\n                sample["status"] = "PATH_UNRESOLVED"\n                result["errors"].append(\n                    {"split": split, "image_id": sample["image_id"], "code": "PATH_UNRESOLVED"}\n                )\n            else:\n                try:\n                    with Image.open(image_path) as image:\n                        image.load()\n                        sample.update(\n                            status="READ_OK",\n                            width=image.width,\n                            height=image.height,\n                            format=image.format,\n                            mode=image.mode,\n                        )\n                    sample["sha256"] = sha256_file(image_path)\n                    sample["bytes"] = image_path.stat().st_size\n                    sample["relative_path"] = str(image_path.relative_to(input_root))\n                    sample_directories.add(image_path.parent)\n                except (OSError, ValueError) as exc:\n                    sample.update(status="READ_FAILED", error_type=type(exc).__name__)\n                    result["errors"].append(\n                        {"split": split, "image_id": sample["image_id"], "code": "READ_FAILED"}\n                    )\n            record["samples"].append(sample)\n        result["splits"][split] = record\n    root = annotations["train"].parent\n    extensions = Counter()\n    mesh_examples = []\n    metadata_examples = []\n    # Inventory only sampled image neighborhoods. Do not imply a complete count.\n    print(f"Inspecting metadata/meshes near {len(sample_directories)} sampled images", flush=True)\n    for sample_directory in sorted(sample_directories):\n        for directory, _, filenames in os.walk(sample_directory):\n            for filename in sorted(filenames):\n                suffix = Path(filename).suffix.lower()\n                extensions[suffix] += 1\n                path = Path(directory) / filename\n                if suffix == ".obj" and len(mesh_examples) < 8:\n                    mesh_examples.append(\n                        {\n                            "path": str(path.relative_to(root)),\n                            "bytes": path.stat().st_size,\n                            "sha256": sha256_file(path),\n                        }\n                    )\n                if (\n                    suffix == ".json"\n                    and path not in annotations.values()\n                    and len(metadata_examples) < 8\n                ):\n                    meta = {"path": str(path.relative_to(root)), "bytes": path.stat().st_size}\n                    if path.stat().st_size < 100_000:\n                        try:\n                            meta["document"] = json.loads(path.read_text(encoding="utf-8"))\n                        except (ValueError, UnicodeError):\n                            meta["status"] = "UNREADABLE_JSON"\n                    metadata_examples.append(meta)\n    result["inventory"] = {\n        "scope": "SAMPLED_IMAGE_NEIGHBORHOODS",\n        "sample_directory_count": len(sample_directories),\n        "sampled_extension_counts": dict(extensions),\n        "mesh_examples": mesh_examples,\n        "metadata_examples": metadata_examples,\n    }\n    if len(group_sets) == 3:\n        overlaps = {\n            f"{a}:{b}": len(group_sets[a] & group_sets[b])\n            for a, b in (("train", "validation"), ("train", "test"), ("validation", "test"))\n        }\n        result["group_overlap_counts"] = overlaps\n        for record in result["splits"].values():\n            record["group_split_verified"] = not any(overlaps.values())\n    result["mount_read_gate"] = (\n        "PASS"\n        if not result["errors"]\n        and all(\n            any(s["status"] == "READ_OK" for s in record["samples"])\n            for record in result["splits"].values()\n        )\n        else "FAIL"\n    )\n    result["correspondence_label_gate"] = "PENDING_REVIEW"\n    result["commercial_license_gate"] = "HOLD_ORIGINAL_RENDERS"\n    result["full_archive_copied"] = False\n    target = output_root / "parcel3d-audit.json"\n    target.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")\n    (output_root / "audit-sha256.txt").write_text(\n        sha256_file(target) + "  parcel3d-audit.json\\n", encoding="ascii"\n    )\n    print(\n        json.dumps(\n            {\n                "mount_read_gate": result["mount_read_gate"],\n                "counts": {k: v["image_count"] for k, v in result["splits"].items()},\n                "outputs": str(output_root),\n            },\n            ensure_ascii=False,\n        )\n    )\n    return result\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input", type=Path, default=Path("/kaggle/input"))\n    parser.add_argument("--output", type=Path, default=Path("/kaggle/working/unihack26-audit"))\n    parser.add_argument("--source-version", type=int, default=1)\n    args = parser.parse_args()\n    audit(args.input, args.output, args.source_version)\n', 'ml/models/__init__.py': '"""Research model candidates; promotion requires the real-data release gates."""\n', 'ml/models/paired_transformer.py': '"""Spatial paired-image candidate, not a validated damage detector.\n\nBoth images must already be in the query coordinate system. Matching/visibility\nis an upstream task. Unknown pixels are excluded from change supervision.\n"""\nfrom __future__ import annotations\n\nimport math\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nDINO_REVISION = "7764ea0f912e53c92e82eb78a2a1631e92725fc8"\nBACKBONES = {\n    "dinov2_vitb14": {"channels": 768, "layers": (2, 5, 8, 11)},\n    "dinov2_vitl14": {"channels": 1024, "layers": (5, 11, 17, 23)},\n}\n\n\nclass LoRALinear(nn.Module):\n    """Low-rank adaptation of a frozen attention projection."""\n\n    def __init__(self, base: nn.Linear, rank: int = 8, alpha: float = 16):\n        super().__init__()\n        if rank < 1:\n            raise ValueError("LORA_RANK_MUST_BE_POSITIVE")\n        self.base = base.requires_grad_(False)\n        self.down = nn.Linear(base.in_features, rank, bias=False)\n        self.up = nn.Linear(rank, base.out_features, bias=False)\n        self.scale = alpha / rank\n        nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5))\n        nn.init.zeros_(self.up.weight)\n\n    def forward(self, x):\n        return self.base(x) + self.up(self.down(x)) * self.scale\n\n\nclass DinoSpatialEncoder(nn.Module):\n    def __init__(self, name="dinov2_vitl14", lora_blocks=0, cache_dir=None):\n        super().__init__()\n        if name not in BACKBONES or not 0 <= lora_blocks <= 4:\n            raise ValueError("UNSUPPORTED_ENCODER_CONFIGURATION")\n        if cache_dir:\n            torch.hub.set_dir(str(cache_dir))\n        self.backbone = torch.hub.load(\n            f"facebookresearch/dinov2:{DINO_REVISION}", name,\n            pretrained=True, trust_repo=True,\n        ).requires_grad_(False)\n        self.name = name\n        self.channels = BACKBONES[name]["channels"]\n        self.layers = BACKBONES[name]["layers"]\n        self.adapted = lora_blocks > 0\n        if self.adapted:\n            for block in self.backbone.blocks[-lora_blocks:]:\n                block.attn.qkv = LoRALinear(block.attn.qkv)\n        self.register_buffer("mean", torch.tensor([.485, .456, .406])[None, :, None, None])\n        self.register_buffer("std", torch.tensor([.229, .224, .225])[None, :, None, None])\n\n    def forward(self, images):\n        if images.shape[-1] % 14 or images.shape[-2] % 14:\n            raise ValueError("ENCODER_SIZE_MUST_BE_MULTIPLE_OF_14")\n        # No stochastic depth in the frozen foundation model. LoRA still gets\n        # gradients in training; no inference-mode tensors enter the decoder.\n        self.backbone.eval()\n        gradients = torch.is_grad_enabled() and self.training and self.adapted\n        with torch.set_grad_enabled(gradients):\n            return list(self.backbone.get_intermediate_layers(\n                (images - self.mean) / self.std, n=self.layers, reshape=True,\n            ))\n\n\nclass CrossViewBlock(nn.Module):\n    """Query patches attend to the aligned reference\'s spatial features."""\n\n    def __init__(self, width=128, heads=4):\n        super().__init__()\n        self.q_norm = nn.LayerNorm(width)\n        self.r_norm = nn.LayerNorm(width)\n        self.attention = nn.MultiheadAttention(width, heads, batch_first=True)\n        self.position = nn.Conv2d(2, width, 1)\n        self.fuse = nn.Sequential(\n            nn.Conv2d(width * 3, width, 1), nn.GroupNorm(8, width), nn.GELU(),\n            nn.Conv2d(width, width, 3, padding=1), nn.GroupNorm(8, width), nn.GELU(),\n        )\n\n    def forward(self, query, reference, valid):\n        b, _, h, w = query.shape\n        yy, xx = torch.meshgrid(\n            torch.linspace(-1, 1, h, device=query.device, dtype=query.dtype),\n            torch.linspace(-1, 1, w, device=query.device, dtype=query.dtype), indexing="ij",\n        )\n        position = self.position(torch.stack([xx, yy])[None]).flatten(2).transpose(1, 2)\n        q = self.q_norm(query.flatten(2).transpose(1, 2))\n        r = self.r_norm(reference.flatten(2).transpose(1, 2))\n        keep = F.interpolate(valid.float(), (h, w), mode="area").flatten(1) >= .5\n        empty = ~keep.any(dim=1)\n        # PyTorch attention is undefined if every key is masked. A dummy key is\n        # enabled only for the numerical operation; its output is then zeroed.\n        safe_keep = keep.clone()\n        safe_keep[empty, 0] = True\n        context, _ = self.attention(\n            q + position, r + position, r,\n            key_padding_mask=~safe_keep, need_weights=False,\n        )\n        context = context.masked_fill(empty[:, None, None], 0)\n        context = context.transpose(1, 2).reshape(b, -1, h, w)\n        return self.fuse(torch.cat([query, query - reference, query - context], 1))\n\n\nclass PairedChangeTransformer(nn.Module):\n    def __init__(self, encoder_channels=1024, width=128, levels=4):\n        super().__init__()\n        if width % 8 or levels < 1:\n            raise ValueError("INVALID_DECODER_CONFIGURATION")\n        self.levels = levels\n        self.projections = nn.ModuleList(nn.Conv2d(encoder_channels, width, 1) for _ in range(levels))\n        self.cross_view = nn.ModuleList(CrossViewBlock(width) for _ in range(levels))\n        self.semantic = nn.Sequential(nn.Conv2d(levels * width, width, 1), nn.GELU())\n        # RGB detail bypass preserves narrow scratches that patch tokens lose.\n        self.detail = nn.Sequential(\n            nn.Conv2d(10, 32, 5, stride=2, padding=2), nn.GroupNorm(8, 32), nn.GELU(),\n            nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.GroupNorm(8, 32), nn.GELU(),\n        )\n        self.decode = nn.Sequential(\n            nn.Conv2d(width + 32, 64, 3, padding=1), nn.GroupNorm(8, 64), nn.GELU(),\n            nn.Conv2d(64, 32, 3, padding=1), nn.GELU(), nn.Conv2d(32, 1, 1),\n        )\n\n    def forward(self, query_features, reference_features, query, reference, valid):\n        if len(query_features) != self.levels or len(reference_features) != self.levels:\n            raise ValueError("FEATURE_LEVEL_COUNT_MISMATCH")\n        shape = query_features[0].shape[-2:]\n        fused = []\n        for q, r, projection, block in zip(\n            query_features, reference_features, self.projections, self.cross_view, strict=True,\n        ):\n            item = block(projection(q), projection(r), valid)\n            fused.append(F.interpolate(item, shape, mode="bilinear", align_corners=False))\n        semantic = self.semantic(torch.cat(fused, 1))\n        detail = self.detail(torch.cat([query, reference, (query - reference).abs(), valid], 1))\n        semantic = F.interpolate(semantic, detail.shape[-2:], mode="bilinear", align_corners=False)\n        logits = self.decode(torch.cat([semantic, detail], 1))\n        return F.interpolate(logits, query.shape[-2:], mode="bilinear", align_corners=False)\n\n\ndef masked_change_loss(logits, target, valid):\n    """Unknown pixels contribute neither gradients nor positive/negative labels."""\n    # Half-precision sums overflow above 65,504 pixels at the target resolutions.\n    logits, target, valid = logits.float(), target.float(), valid.float()\n    bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")\n    bce = (bce * valid).sum() / valid.sum().clamp_min(1)\n    p, t = logits.sigmoid() * valid, target * valid\n    dims = (1, 2, 3)\n    dice = 1 - (2 * (p * t).sum(dims) + 1) / (p.sum(dims) + t.sum(dims) + 1)\n    observed = valid.sum(dims) > 0\n    return bce + (dice * observed).sum() / observed.sum().clamp_min(1)\n', 'ml/train/__init__.py': '', 'ml/train/upgrade_profile.py': '"""Bounded GPU feasibility measurements; does not produce accuracy claims."""\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport json\nimport platform\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image\n\nfrom ml.data.parcel3d_audit import find_annotations, within_root\nfrom ml.models.paired_transformer import DinoSpatialEncoder, PairedChangeTransformer, masked_change_loss\n\n\ndef file_sha(path):\n    sha = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            sha.update(chunk)\n    return sha.hexdigest()\n\n\ndef run(input_root, output_root, max_seconds=1200):\n    output_root = Path(output_root)\n    output_root.mkdir(parents=True, exist_ok=True)\n    started = time.monotonic()\n    torch.manual_seed(26)\n    torch.set_num_threads(2)\n    report = {"purpose": "FEASIBILITY_ONLY_NOT_TRAINING_OR_ACCURACY",\n              "python": platform.python_version(), "torch": torch.__version__,\n              "cuda_available": torch.cuda.is_available(), "measurements": []}\n\n    def save():\n        report["elapsed_seconds"] = time.monotonic() - started\n        (output_root / "profile.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n\n    if not torch.cuda.is_available():\n        report["status"] = "BLOCKED_NO_CUDA"\n        save()\n        raise RuntimeError("GPU_REQUIRED_NO_CPU_FALLBACK")\n    report["gpu"] = torch.cuda.get_device_name(0)\n    report["gpu_total_bytes"] = torch.cuda.get_device_properties(0).total_memory\n    annotations = find_annotations(Path(input_root))["train"]\n    source = json.loads(annotations.read_text())["images"][0]\n    source_path = within_root(annotations.parent, source["file_name"])\n    with Image.open(source_path) as image:\n        source_image = image.convert("RGB")\n    report["source_image_sha256"] = file_sha(source_path)\n    report["source_split"] = "OFFICIAL_TRAIN_ONLY"\n    plans = [("dinov2_vitb14", 392, 0), ("dinov2_vitl14", 392, 0),\n             ("dinov2_vitl14", 392, 4), ("dinov2_vitl14", 518, 0)]\n    for name, size, lora_blocks in plans:\n        if time.monotonic() - started >= max_seconds:\n            report["status"] = "PARTIAL_TIME_BUDGET"\n            break\n        row = {"encoder": name, "size": size, "lora_blocks": lora_blocks, "batch_pairs": 1}\n        encoder = decoder = optimizer = None\n        try:\n            encoder = DinoSpatialEncoder(name, lora_blocks, "/kaggle/temp/kc-model-cache").cuda().train()\n            decoder = PairedChangeTransformer(encoder.channels).cuda().train()\n            params = [p for module in (encoder, decoder) for p in module.parameters() if p.requires_grad]\n            optimizer = torch.optim.AdamW(params, lr=1e-4)\n            scaler = torch.amp.GradScaler("cuda")\n            array = np.asarray(source_image.resize((size, size)), dtype=np.float32).copy() / 255\n            reference = torch.from_numpy(array).permute(2, 0, 1)[None].cuda()\n            query = (reference * .9 + .025).clamp(0, 1)\n            valid = torch.ones_like(query[:, :1])\n            target = torch.zeros_like(valid)\n            query[:, :, size // 3:size // 3 + 5, size // 4:3 * size // 4] = .25\n            target[:, :, size // 3:size // 3 + 5, size // 4:3 * size // 4] = 1\n            row["trainable_parameters"] = sum(p.numel() for p in params)\n            row["encoder_parameters"] = sum(p.numel() for p in encoder.parameters())\n            torch.cuda.reset_peak_memory_stats()\n            timings, losses = [], []\n            for step in range(6):\n                optimizer.zero_grad(set_to_none=True)\n                torch.cuda.synchronize()\n                step_start = time.monotonic()\n                with torch.autocast("cuda", dtype=torch.float16):\n                    features = encoder(torch.cat([query, reference], 0))\n                    q, r = [x[:1] for x in features], [x[1:] for x in features]\n                    logits = decoder(q, r, query, reference, valid)\n                    value = masked_change_loss(logits, target, valid)\n                if not torch.isfinite(value):\n                    raise RuntimeError("NONFINITE_PROFILE_LOSS")\n                scaler.scale(value).backward()\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(params, 1.)\n                scaler.step(optimizer)\n                scaler.update()\n                torch.cuda.synchronize()\n                if step >= 2:\n                    timings.append(time.monotonic() - step_start)\n                    losses.append(float(value.detach()))\n            weight = Path(torch.hub.get_dir()) / "checkpoints" / f"{name}_pretrain.pth"\n            row.update(status="PASS", mean_step_seconds=float(np.mean(timings)),\n                       p95_step_seconds=float(np.percentile(timings, 95)),\n                       peak_allocated_bytes=torch.cuda.max_memory_allocated(),\n                       peak_reserved_bytes=torch.cuda.max_memory_reserved(),\n                       backbone_sha256=file_sha(weight), finite_losses=losses,\n                       note="Single repeated synthetic pair; losses are NOT a benchmark.")\n        except torch.cuda.OutOfMemoryError:\n            row["status"] = "OOM"\n        except Exception as exc:\n            row.update(status="ERROR", error_type=type(exc).__name__, message=str(exc)[:300])\n        finally:\n            # The notebook exits after profiling; weights stay in temporary cache\n            # and no trained checkpoint is exported or promoted.\n            encoder = decoder = optimizer = None\n            params = features = q = r = logits = value = None\n            gc.collect()\n            torch.cuda.empty_cache()\n        report["measurements"].append(row)\n        save()\n        print(json.dumps(row), flush=True)\n    report.setdefault("status", "COMPLETE" if all(r["status"] == "PASS" for r in report["measurements"]) else "REVIEW_REQUIRED")\n    save()\n    return report\n'}
root=Path('/kaggle/working/_src')
for name,source in SOURCES.items():
 p=root/name
 p.parent.mkdir(parents=True,exist_ok=True)
 p.write_text(source,encoding='utf-8')
sys.path.insert(0,str(root))


In [ ]:
from ml.train.upgrade_profile import run
result = run('/kaggle/input', '/kaggle/working/profile', max_seconds=1200)
Path('/kaggle/working/source-hashes.json').write_text(json.dumps({'ml/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/data/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/data/parcel3d_audit.py': '08ddf5f683f5b23c8364228d53f8b3883518465c3e2d66a64bf3ff9888f46b4e', 'ml/models/__init__.py': '970a22e9a2f8997752da0cb86ff07508de56946ba32d03e4d2d9b92a35f29399', 'ml/models/paired_transformer.py': 'd211fafb77b78b555fc9863c85178962dce3b1a39851343b82befb63663154e1', 'ml/train/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/train/upgrade_profile.py': 'ce5a81d6feb0a9694cd207c003165d4e5eef00cfbfbecc9d836c06615dee3b31'},indent=2))
